# R5-3 Edge Feasibility Benchmark — FP16 Only Safe Rerun

This notebook runs only the FP16 benchmark and SQLite payload-buffering test. It writes results to a new timestamped folder and does not overwrite previous outputs.


In [ ]:
!pip uninstall -y torchao

In [ ]:
# ============================================================
# Cell 0: Install / imports
# R5-3 FP16-only safe rerun version
# ============================================================
!pip -q install "transformers>=4.44.0" accelerate peft datasets tqdm pandas psutil

import os, json, re, time, gc
import numpy as np
import pandas as pd
import psutil
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# Cell 1: Mount Drive + load adapter config
# FP16-only safe rerun: every run writes to a NEW timestamped folder.
# ============================================================
ADAPTER_DIR = "./qwen3_cold_chain_s3_driver_lora"

# Old/original result folder is kept unchanged for comparison.
PREVIOUS_ARTIFACT_DIR = "outputs/edge_feasibility_outputs"

# New rerun folder. This prevents replacing any previous code/output.
BASE_ARTIFACT_ROOT = "outputs/edge_feasibility_outputs_fp16_reruns"
RUN_TAG = time.strftime("fp16_rerun_%Y%m%d_%H%M%S")
ARTIFACT_DIR = os.path.join(BASE_ARTIFACT_ROOT, RUN_TAG)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

print("ADAPTER_DIR exists?", os.path.isdir(ADAPTER_DIR))
assert os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_config.json")), "adapter_config.json not found"
assert os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")), "adapter_model.safetensors not found"

cfg_path = os.path.join(ADAPTER_DIR, "adapter_config.json")
with open(cfg_path, "r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)
BASE_MODEL = adapter_cfg.get("base_model_name_or_path", "Qwen/Qwen3-0.6B")

print("Using BASE_MODEL =", BASE_MODEL)
print("PREVIOUS_ARTIFACT_DIR =", PREVIOUS_ARTIFACT_DIR)
print("NEW ARTIFACT_DIR =", ARTIFACT_DIR)

# Save a small run-info file for traceability.
run_info = {
    "run_tag": RUN_TAG,
    "artifact_dir": ARTIFACT_DIR,
    "previous_artifact_dir": PREVIOUS_ARTIFACT_DIR,
    "adapter_dir": ADAPTER_DIR,
    "base_model": BASE_MODEL,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
with open(os.path.join(ARTIFACT_DIR, "run_info.json"), "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
print("Saved run_info.json")


In [ ]:
# ============================================================


# ============================================================
PROMPT_TEMPLATE = """You are an assistant that generates short, practical alerts for refrigerated-truck drivers transporting strawberries.

Your job:
- Turn a structured 1-hour risk summary into a 4-line driver message.

Hard rules:
- Use this fixed format and these exact line headers:
  Line 1: Status: ...
  Line 2: Temp pattern: ...
  Line 3: Data quality: ...
  Line 4: Action: ...
- Use simple, practical language suitable for a busy driver.
  - Short sentences.
  - No technical terms (no "respiration rate", "variance", etc.).
- Focus on:
  - what is happening in this 1-hour window,
  - and what the driver should do now.
- Map risk level to Status:
  - R0 → "Status: Normal – ..."
  - R1 → "Status: Warning – ..."
  - R2 → "Status: High risk – ..."
- Map confidence level to Data quality:
  - high confidence → "Data quality: High – ..."
  - medium confidence → "Data quality: Medium – ..."
  - low confidence → "Data quality: Low – ..."
- Map the Action line as follows, and ALWAYS include a dash (–) after the main verb:
  - For R0 (normal): use "Action: Maintain – ..." with a short instruction
    (e.g., keep current settings, drive as usual).
  - For R1 (warning): use "Action: Review – ..." with a short instruction
    (e.g., reduce long door openings, check set-point and airflow).
  - For R2 (high risk): use "Action: Act now – ..." with a short instruction
    (e.g., lower the fridge set-point, keep doors closed, check cooling unit or cargo).
  - Do NOT write "Action: ... with ...". Always use the pattern "Action: <verb> – <details>".
- Use the pattern of temperatures and variation:
  - If standard deviation is low and conditions are consistent → talk about "stable" or "steady" pattern.
  - If variation is large or locations differ a lot → talk about "uneven", "mixed", or "large differences".
- Do NOT mention internal model details or equations.
- Do NOT output the summary again. Only output the 4 driver lines.
- Output EXACTLY 4 lines and NOTHING else.
- Do NOT write any labels like "Answer:", "Response:", or explanations.
- Do NOT use JSON, curly braces {{ }}, or code blocks.
- Do NOT repeat the summary.
- Do NOT invent new numbers or time durations that are not clearly in the summary.
- If you mention durations, always use minutes (e.g., "30 minutes"), never seconds.

Style rules:
- Prefer qualitative wording like "long time above 4°C" or
  "short period below 0°C" instead of exact counts of minutes.
- Only mention exact durations (e.g. "30 minutes") if that number
  is clearly written in the summary.
- Do NOT use patterns like "60 minutes above 4°C and 0 minutes below 0°C".
- For warm severe cases (R2, temperatures clearly above 4°C),
  prefer phrases like:
  - "long time above 4°C"
  - "strong warm peak"
  - "moderate temperature variation"
  - "large temperature swings"
  instead of repeating numeric statistics.

Here are examples:

Example 1 – R0, low risk, stable and in-range, high confidence
Summary:
In this 1-hour window, the assessed risk level is R0 (low). All measured temperatures stay between 0°C and 4°C with only very small variation, and there are no excursions into freezing or excessive warming. This pattern matches the recommended storage range for strawberries and should support normal shelf-life. Sensor coverage is complete and consistent, so the assessment is made with high confidence and the temperature behaviour is stable.

Driver message:
Status: Normal – safe and stable temperature.
Temp pattern: In-range – temperatures stayed between 0–4°C with only small variation.
Data quality: High – sensors give a consistent and complete cold pattern.
Action: Maintain – keep current fridge settings and drive as usual.


Example 2 – R1, moderate risk, short deviations, high confidence
Summary:
In this 1-hour window, the assessed risk level is R1 (moderate). Temperatures stay close to the 0–4°C target range but drift slightly outside it for short periods, for example briefly above 4°C or just below 0°C. These deviations are not long enough to reach severe-risk conditions, but they may cause quality loss if repeated many times. Temperature variation is moderate but not extreme, and all sensors report normally, so the assessment is made with high confidence.

Driver message:
Status: Warning – minor temperature deviation.
Temp pattern: Mixed – short time just outside 0–4°C with moderate variation.
Data quality: High – sensors show the pattern clearly.
Action: Review – reduce long door openings and confirm that set-point and airflow are correct.


Example 3 – R2, severe high temperature, long time warm, stable warm pattern, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are well above 4°C for almost the entire hour and there is at least one strong warm peak clearly above 10°C. The mean temperature is high and the variation is not large, so the load is consistently too warm rather than just briefly spiking. This warm exposure can strongly shorten the remaining shelf-life of strawberries. Sensor coverage is high and consistent across locations, so the assessment and the warm pattern are both considered reliable.

Driver message:
Status: High risk – cargo too warm.
Temp pattern: Over-warm – almost the whole hour well above 4°C with a strong warm peak and stable warm pattern.
Data quality: High – multiple sensors agree on this warm condition.
Action: Act now – lower the fridge set-point, keep doors closed, and check the cooling unit and airflow.


Example 4 – R2, severe high temperature, uneven and with missing data, medium confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are above 4°C for a long time and there is a strong warm peak above 10°C, indicating that at least part of the load is over-warm. However, some sensor readings are missing or incomplete, and the variation between locations is quite large. This means parts of the cargo may be much warmer than others, but the exact distribution is uncertain. Because of the missing data and uneven pattern, the assessment is made with only medium confidence.

Driver message:
Status: High risk – cargo likely too warm.
Temp pattern: Over-warm and uneven – long time above 4°C with a strong warm peak and large differences inside the load.
Data quality: Medium – some sensor readings are missing or incomplete.
Action: Act now – lower the set-point, keep doors closed, and, if possible, check the cargo condition manually.


Example 5 – R2, severe risk, warm for too long without extreme peak, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures stay above 4°C for around half an hour, while the rest of the period is close to the lower limit of the recommended range. Even without an extreme peak above 10°C, this prolonged moderate warming can still accelerate quality loss, especially if similar episodes repeat during the trip. Temperature variation is moderate and sensor readings are complete across time and locations, so the warm period is clearly visible and the assessment is made with high confidence.

Driver message:
Status: High risk – cargo warm for too long.
Temp pattern: Over-warm – about half an hour above 4°C and the rest near the lower limit, with moderate variation.
Data quality: High – sensors show a clear warm period.
Action: Act now – shorten door-opening times, check fridge capacity, and consider a slightly lower set-point.


Example 6 – R2, severe risk, long period near/below 0°C, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). The maximum temperature stays at or below 4°C, but minimum values drop close to or slightly below 0°C, and the cargo spends around 30 minutes at or below 0°C. This over-cooling near the freezing point can cause chilling injury or partial freezing if it occurs repeatedly. The temperature pattern is fairly stable and cold, and all sensors report consistently, so the assessment is made with high confidence.

Driver message:
Status: High risk – cargo too cold.
Temp pattern: Over-cold – around 30 minutes close to or below 0°C with a stable cold pattern.
Data quality: High – sensors show a consistent cold trend.
Action: Act now – raise the set-point slightly, avoid very strong cooling, and watch the coldest positions.


Example 7 – R2, severe risk, possible freezing damage, high confidence
Summary:
In this 1-hour window, the assessed risk level is R2 (severe). Temperatures are below 0°C for most of the hour, and the coldest locations drop to around or below −1°C. This cold exposure is close to or beyond the freezing point of strawberries and can cause irreversible freezing damage, such as darkening, watery texture, and collapse after re-warming. The cold pattern is steady and several sensors agree on this behaviour, so the assessment is made with high confidence.

Driver message:
Status: High risk – possible freezing damage.
Temp pattern: Freezing – long time below 0°C with the coldest spots near or below −1°C and a steady cold pattern.
Data quality: High – several sensors confirm this freezing behaviour.
Action: Act now – increase the set-point, avoid strong cooling, and flag this batch for a quality check at destination.

----------------
Now a new case:
Summary:
{PUT_SUMMARY_TEXT_HERE}

Please generate ONLY the 4-line driver message in the required format:
Line 1: Status: ...
Line 2: Temp pattern: ...
Line 3: Data quality: ...
Line 4: Action: ...
"""

In [ ]:
# ============================================================




# ============================================================
def clean_driver_message(raw_text: str) -> str:
    required_prefixes = ["Status:", "Temp pattern:", "Data quality:", "Action:"]
    collected = {p: None for p in required_prefixes}

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            continue

        for p in required_prefixes:
            if collected[p] is None and line.startswith(p):
                collected[p] = line
                break

        if all(collected[p] is not None for p in required_prefixes):
            break

    if all(collected[p] is not None for p in required_prefixes):
        return "\n".join([collected[p] for p in required_prefixes])
    else:
        return raw_text.strip()


_FMT_OK_RE = re.compile(
    r"(?is)Status:\s*(.*?)\s*Temp pattern:\s*(.*?)\s*Data quality:\s*(.*?)\s*Action:\s*(.*)"
)

def is_format_ok(msg: str) -> bool:
    return isinstance(msg, str) and bool(_FMT_OK_RE.search(msg))


In [ ]:
# ============================================================
# Cell 6: Load FP16 base model + LoRA adapter
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=False,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(model, ADAPTER_DIR)

MERGE_LORA = True
if MERGE_LORA:
    model = model.merge_and_unload()

model.eval()
print("FP16 model ready")


In [ ]:
# ============================================================


# 1) prompt_for_model = prompt + "\n"




# ============================================================

MAX_PROMPT_TOKENS = 2048

def make_runner(model, tokenizer, prompt_template: str, max_prompt_tokens: int = 2048):

    fmt_re = re.compile(r"(?is)Status:.*?Temp pattern:.*?Data quality:.*?Action:")

    def encode_left_truncate(prompt_for_model: str):
        enc = tokenizer(prompt_for_model, add_special_tokens=False, return_tensors="pt")
        input_ids = enc["input_ids"][0]
        if input_ids.numel() > max_prompt_tokens:
            input_ids = input_ids[-max_prompt_tokens:]
        attn = torch.ones_like(input_ids)
        return {
            "input_ids": input_ids.unsqueeze(0).to(model.device),
            "attention_mask": attn.unsqueeze(0).to(model.device),
        }

    @torch.no_grad()
    def generate_once(summary: str, max_new_tokens: int = 160, min_new_tokens: int = 32, debug: bool = False):
        prompt = prompt_template.format(PUT_SUMMARY_TEXT_HERE=str(summary))
        prompt_for_model = prompt + "\n"
        inputs = encode_left_truncate(prompt_for_model)
        input_len = int(inputs["input_ids"].shape[1])

        out = model.generate(
            **inputs,
            do_sample=False,  # greedy
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True,
        )

        gen_ids = out[0][input_len:]
        raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        cleaned = clean_driver_message(raw)

        meta = {
            "input_len": input_len,
            "gen_tokens": int(gen_ids.numel()),
            "raw_len_chars": len(raw),
            "format_ok": bool(fmt_re.search(cleaned)),
        }

        if debug:
            print(f"[debug] input_len={meta['input_len']} gen_tokens={meta['gen_tokens']} format_ok={meta['format_ok']}")
            print("[debug] raw head:\n", raw[:400])

        return cleaned, meta

    def run(summary: str, max_new_tokens: int = 256, retries: int = 1, debug: bool = False):
        last_msg, last_meta = None, None
        for attempt in range(retries + 1):
            msg, meta = generate_once(summary, max_new_tokens=max_new_tokens, min_new_tokens=32, debug=debug)
            meta.update({"attempt": attempt, "retried": attempt > 0})

            last_msg, last_meta = msg, meta
            if meta["format_ok"]:
                return msg, meta


        return last_msg, last_meta

    return run

# ============================================================
# Cell 5: Create runner
# ============================================================
MAX_PROMPT_TOKENS = 2048
run_model = make_runner(model, tokenizer, PROMPT_TEMPLATE, max_prompt_tokens=MAX_PROMPT_TOKENS)
print("runner ready")


In [ ]:
# ============================================================
# Cell 6: Prepare benchmark data
# ============================================================
SPLIT_FOR_BENCH = "s4"
N_BENCH = 1000

dataset_name = "NifferLi/cold-chain-strawberry-sensors"
ds = load_dataset(dataset_name)
bench_ds = ds[SPLIT_FOR_BENCH].filter(lambda x: bool(str(x.get("summary_text", "")).strip()))
N_BENCH = min(N_BENCH, len(bench_ds))

rng = np.random.RandomState(0)
idx = rng.choice(len(bench_ds), size=N_BENCH, replace=False)
bench_summaries = [bench_ds[int(i)]["summary_text"] for i in idx]

print("Bench split:", SPLIT_FOR_BENCH, "| N_BENCH:", len(bench_summaries))
print("Example head:\n", bench_summaries[0][:300])


In [ ]:
# ============================================================
# Cell 7: Benchmark utility functions (R5-3 enhanced)
# ============================================================
def fmt_seconds(sec: float) -> str:
    sec = int(round(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

def bytes_to_gb(x: int) -> float:
    return float(x) / (1024**3)

LATENCY_TARGET_60S_MS = 60_000
LATENCY_TARGET_10MIN_MS = 10 * 60 * 1000

def _pct_exceed(lat_ms, threshold_ms):
    arr = np.asarray(lat_ms, dtype=float)
    if arr.size == 0:
        return None
    return float(np.mean(arr > threshold_ms) * 100.0)

def benchmark_inference(run_fn, summaries, warmup=5, desc="bench"):
    proc = psutil.Process(os.getpid())

    for i in range(min(warmup, len(summaries))):
        _ = run_fn(summaries[i], max_new_tokens=160, retries=1, debug=False)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    lat_ms = []
    messages = []
    format_ok_list = []
    total_gen_tokens = 0
    total_time_s = 0.0
    peak_rss = 0
    ok_cnt = 0

    pbar = tqdm(summaries, desc=desc, unit="sample")
    t_start = time.perf_counter()

    for k, s in enumerate(pbar, start=1):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        msg, meta = run_fn(s, max_new_tokens=160, retries=1, debug=False)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        dt = t1 - t0
        latency_ms = dt * 1000.0
        lat_ms.append(latency_ms)
        messages.append(msg)
        is_ok = bool(meta.get("format_ok") is True)
        format_ok_list.append(is_ok)
        total_time_s += dt
        total_gen_tokens += int(meta.get("gen_tokens") or 0)
        ok_cnt += int(is_ok)
        peak_rss = max(peak_rss, proc.memory_info().rss)

        if k % 20 == 0:
            p50 = float(np.percentile(lat_ms, 50))
            pmax = float(np.max(lat_ms))
            ok_rate = ok_cnt / k
            elapsed = time.perf_counter() - t_start
            pbar.set_postfix({
                "p50(ms)": round(p50, 1),
                "max(ms)": round(pmax, 1),
                "ok%": round(ok_rate*100, 1),
                "elapsed": fmt_seconds(elapsed),
            })

    peak_vram = None
    if torch.cuda.is_available():
        peak_vram = int(torch.cuda.max_memory_allocated())

    return {
        "n": len(lat_ms),
        "latencies_ms": [float(x) for x in lat_ms],
        "messages": messages,
        "format_ok_list": format_ok_list,
        "lat_p50_ms": float(np.percentile(lat_ms, 50)),
        "lat_p95_ms": float(np.percentile(lat_ms, 95)),
        "lat_p99_ms": float(np.percentile(lat_ms, 99)),
        "lat_max_ms": float(np.max(lat_ms)),
        "exceed_60s_pct": _pct_exceed(lat_ms, LATENCY_TARGET_60S_MS),
        "exceed_10min_pct": _pct_exceed(lat_ms, LATENCY_TARGET_10MIN_MS),
        "throughput_tokens_per_s": (total_gen_tokens / total_time_s) if total_time_s > 0 else None,
        "peak_ram_bytes": int(peak_rss),
        "peak_vram_bytes": int(peak_vram) if peak_vram is not None else None,
        "format_ok_rate": ok_cnt / len(lat_ms) if len(lat_ms) else None,
        "total_time_s": total_time_s,
        "total_gen_tokens": total_gen_tokens,
    }

def benchmark_sqlite_buffer(payloads, db_path="outputs/edge_buffer.db"):
    import sqlite3
    if os.path.exists(db_path):
        os.remove(db_path)

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS alerts(
      id INTEGER PRIMARY KEY AUTOINCREMENT,
      ts REAL,
      payload TEXT,
      uploaded INTEGER DEFAULT 0
    )
    """)
    conn.commit()

    ts = time.time()
    t0 = time.perf_counter()
    cur.executemany("INSERT INTO alerts(ts,payload,uploaded) VALUES(?,?,0)", [(ts, p) for p in payloads])
    conn.commit()
    t1 = time.perf_counter()
    write_s = (t1 - t0)

    t2 = time.perf_counter()
    cur.execute("SELECT id FROM alerts WHERE uploaded=0")
    ids = [r[0] for r in cur.fetchall()]
    cur.executemany("UPDATE alerts SET uploaded=1 WHERE id=?", [(i,) for i in ids])
    conn.commit()
    t3 = time.perf_counter()
    flush_s = (t3 - t2)

    conn.close()

    n = len(payloads)
    return {
        "buffer_n": n,
        "sqlite_write_ms_total": write_s * 1000.0,
        "sqlite_write_ms_per_record": (write_s * 1000.0 / n) if n > 0 else None,
        "buffer_flush_ms": flush_s * 1000.0,
        "flush_records_per_s": (n / flush_s) if flush_s > 0 else None,
    }

def json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (bool, np.bool_)):
        return bool(obj)
    return obj

def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=json_safe)
    print(" Saved:", path)


def load_json(path):
    assert os.path.isfile(path), f" Missing file: {path}"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


In [ ]:
# ============================================================
# Cell 8: Run FP16 inference benchmark + FP16 payloads/SQLite
# R5-3 FP16-only version
# Note: SQLite uses the 1,000 messages already generated during the inference benchmark.
# This avoids running the FP16 model twice while preserving the payload-buffering test.
# ============================================================
PRECISION_NAME = "FP16"
N_BUFFER_PAYLOAD = 1000

print("==== [1/4] FP16 inference benchmark ====")
r_fp16 = benchmark_inference(run_model, bench_summaries, warmup=5, desc="FP16 benchmark")
print({k: v for k, v in r_fp16.items() if k not in ["latencies_ms", "messages", "format_ok_list"]}, "\n")

# Save latency trace and generated payloads immediately
trace_fp16 = pd.DataFrame({
    "Precision": PRECISION_NAME,
    "sample_index": list(range(len(r_fp16["latencies_ms"]))),
    "latency_ms": r_fp16["latencies_ms"],
    "exceed_60s": [lat > LATENCY_TARGET_60S_MS for lat in r_fp16["latencies_ms"]],
    "exceed_10min": [lat > LATENCY_TARGET_10MIN_MS for lat in r_fp16["latencies_ms"]],
    "format_ok": r_fp16["format_ok_list"],
})
trace_path = os.path.join(ARTIFACT_DIR, "edge_latency_trace_fp16.csv")
trace_fp16.to_csv(trace_path, index=False)
print("Saved FP16 latency trace:", trace_path)

payloads_fp16_df = pd.DataFrame({
    "Precision": PRECISION_NAME,
    "sample_index": list(range(len(r_fp16["messages"]))),
    "payload": r_fp16["messages"],
    "format_ok": r_fp16["format_ok_list"],
})
payload_path = os.path.join(ARTIFACT_DIR, "edge_payloads_fp16.csv")
payloads_fp16_df.to_csv(payload_path, index=False)
print("Saved FP16 payloads:", payload_path)

metrics_fp16 = {k: v for k, v in r_fp16.items() if k not in ["latencies_ms", "messages", "format_ok_list"]}
metrics_path = os.path.join(ARTIFACT_DIR, "edge_metrics_fp16_inference.json")
save_json(metrics_path, metrics_fp16)

print("==== [2/4] FP16 payloads + sqlite ====")
payloads_fp16 = r_fp16["messages"][:min(N_BUFFER_PAYLOAD, len(r_fp16["messages"]))]
b_fp16 = benchmark_sqlite_buffer(payloads_fp16, db_path="outputs/edge_buffer_fp16.db")
print(b_fp16, "\n")
save_json(os.path.join(ARTIFACT_DIR, "edge_metrics_fp16_sqlite.json"), b_fp16)

print("FP16-only benchmark finished.")


# Optional: keep the model in memory if you want to inspect examples.
# To release GPU memory after finishing, uncomment the following lines.
# try:
#     del model, tokenizer, run_model
# except NameError:
#     pass
# gc.collect()
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
# print("FP16-only run complete.")


# FP16-only final table
fp16_row = {
    "Setting": "FP16",
    "N": metrics_fp16.get("n"),
    "Latency p50 (ms)": metrics_fp16.get("lat_p50_ms"),
    "Latency p95 (ms)": metrics_fp16.get("lat_p95_ms"),
    "Latency p99 (ms)": metrics_fp16.get("lat_p99_ms"),
    "Max latency (ms)": metrics_fp16.get("lat_max_ms"),
    "Exceed 60s (%)": metrics_fp16.get("exceed_60s_pct"),
    "Exceed 10min (%)": metrics_fp16.get("exceed_10min_pct"),
    "Throughput (tokens/s)": metrics_fp16.get("throughput_tokens_per_s"),
    "Peak RAM (GB)": bytes_to_gb(metrics_fp16.get("peak_ram_bytes", 0)),
    "Peak VRAM (GB)": bytes_to_gb(metrics_fp16.get("peak_vram_bytes", 0)),
    "Format OK rate": metrics_fp16.get("format_ok_rate"),
    "SQLite write (ms/record)": b_fp16.get("sqlite_write_ms_per_record"),
}

df_fp16 = pd.DataFrame([fp16_row])
for col in df_fp16.columns:
    if col != "Setting":
        df_fp16[col] = pd.to_numeric(df_fp16[col], errors="coerce").round(3)

display(df_fp16)
summary_csv = os.path.join(ARTIFACT_DIR, "edge_metrics_fp16_only_final_table.csv")
summary_xlsx = os.path.join(ARTIFACT_DIR, "edge_metrics_fp16_only_final_table.xlsx")
df_fp16.to_csv(summary_csv, index=False)
df_fp16.to_excel(summary_xlsx, index=False)
print("Saved FP16-only final table:")
print(summary_csv)
print(summary_xlsx)
